# Deep Learning WCE Classification Project

**Objective:** Build and compare deep learning models to detect GI diseases from Wireless Capsule Endoscopy (WCE) images, focusing on handling class imbalance.

**Dataset:** [Kvasir-Capsule](https://osf.io/dv2ag/)

**Models:** EfficientNetB0, MobileNetV2, ResNet101V2

---

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')

from src.data_utils import load_image_paths, get_class_distribution, plot_class_distribution, load_image
from src.sampling import undersample, oversample_augment, show_augmentation_samples
from src.preprocessing import preprocess_dataset
from src.models import build_model, print_model_summary, build_all_models
from src.training import compile_model, train_model, plot_training_history, plot_lr_schedule
from src.evaluation import evaluate_model, plot_confusion_matrix, build_comparison_table, display_comparison

import warnings
warnings.filterwarnings('ignore')

print('All imports successful!')

---
## Task 1 — Understand the Imbalance

Load the Kvasir-Capsule dataset and analyze class distribution.

In [ ]:
# Load image paths from dataset
class_images = load_image_paths()
distribution = get_class_distribution(class_images)

print('Classes found:', len(distribution))
for cls, count in sorted(distribution.items(), key=lambda x: -x[1]):
    print(f'  {cls:30s} → {count:>6,} images')

In [ ]:
# Plot original class distribution
plot_class_distribution(
    distribution,
    title='Kvasir-Capsule — Original Class Distribution',
    save_path='../outputs/class_distribution_original.png'
)

### Analysis — Why Imbalance is Dangerous

*Write your 8-10 line analysis here about why class imbalance is dangerous in medical diagnosis.*

> **Key points to cover:**
> - A model that always predicts the majority class gets high accuracy but misses critical conditions
> - In medical diagnosis, false negatives (missing a disease) are far more dangerous than false positives
> - Standard accuracy is a misleading metric for imbalanced data
> - Rare conditions like bleeding or ulcers are clinically the most important to detect

---
## Task 2 — Under-Sampling Majority Classes

In [ ]:
# Under-sample: cap majority classes at 200 images
THRESHOLD = 200
undersampled = undersample(class_images, threshold=THRESHOLD)
under_dist = get_class_distribution(undersampled)

print(f'\nOriginal total: {sum(distribution.values()):,}')
print(f'After under-sampling: {sum(under_dist.values()):,}')
print(f'Data removed: {sum(distribution.values()) - sum(under_dist.values()):,}')

In [ ]:
# Plot under-sampled distribution
plot_class_distribution(
    under_dist,
    title=f'Class Distribution After Under-Sampling (threshold={THRESHOLD})',
    save_path='../outputs/class_distribution_undersampled.png'
)

---
## Task 3 — Over-Sampling via Augmentation

In [ ]:
# Over-sample minority classes in the undersampled data
balanced = oversample_augment(undersampled, threshold=THRESHOLD)
balanced_dist = get_class_distribution(balanced)

print(f'After augmentation: {sum(balanced_dist.values()):,}')
for cls, count in sorted(balanced_dist.items()):
    print(f'  {cls:30s} → {count:>4} images')

In [ ]:
# Show augmentation examples for a minority class
minority_class = min(distribution, key=distribution.get)
sample_img = class_images[minority_class][0]
show_augmentation_samples(sample_img, n_samples=5, save_path='../outputs/augmentation_samples.png')

---
## Task 4 — Pre-Processing

In [ ]:
# Preprocess the balanced dataset (under-sampled + augmented)
data = preprocess_dataset(balanced)

X_train, y_train = data['X_train'], data['y_train']
X_val,   y_val   = data['X_val'],   data['y_val']
X_test,  y_test  = data['X_test'],  data['y_test']
class_names = data['class_names']
num_classes = len(class_names)

print(f'\nNumber of classes: {num_classes}')
print(f'Classes: {class_names}')

---
## Task 5 — Transfer Learning Models

In [ ]:
# Build all three models
models = build_all_models(num_classes=num_classes, dropout_rate=0.3, freeze_ratio=0.8)

---
## Task 6 — Learning Rate Schedules

In [ ]:
# Visualize cosine decay LR schedule
plot_lr_schedule(
    initial_lr=1e-3,
    total_epochs=30,
    steps_per_epoch=len(X_train) // 32,
    save_path='../outputs/lr_cosine_decay.png'
)

---
## Task 7 — Train & Compare Under 3 Settings

| Setting | Description |
|:--------|:------------|
| Baseline | Raw imbalanced data, no handling |
| Under-sampling | Majority classes trimmed to threshold |
| Under-sampling + Augmentation | Balanced via both techniques |

In [ ]:
# ── Setting 1: Baseline (imbalanced) ──
# Preprocess raw imbalanced data
data_baseline = preprocess_dataset(class_images)

# ── Setting 2: Under-sampled only ──
data_under = preprocess_dataset(undersampled)

# ── Setting 3: Under-sampled + Augmented (already done above) ──
# data is already preprocessed as `data`

settings = {
    'Baseline': data_baseline,
    'Under-sampling': data_under,
    'Under-sampling + Augmentation': data,
}

print('Settings prepared:', list(settings.keys()))

In [ ]:
# Train all models under all settings and collect results
EPOCHS = 30
BATCH_SIZE = 32
all_results = {}

for setting_name, setting_data in settings.items():
    print(f'\n{"="*60}')
    print(f'  Setting: {setting_name}')
    print(f'{"="*60}')
    
    all_results[setting_name] = {}
    
    for arch in ['EfficientNetB0', 'MobileNetV2', 'ResNet101V2']:
        print(f'\n  → Training {arch}...')
        
        model = build_model(arch, num_classes=num_classes)
        model = compile_model(model, lr_strategy='reduce_on_plateau')
        
        history = train_model(
            model,
            setting_data['X_train'], setting_data['y_train'],
            setting_data['X_val'], setting_data['y_val'],
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            save_name=f'{setting_name}_{arch}',
        )
        
        # Plot training curves
        plot_training_history(
            history,
            title=f'{arch} — {setting_name}',
            save_path=f'../outputs/{setting_name.replace(" ", "_")}_{arch}_history.png'
        )
        
        # Evaluate on test set
        metrics = evaluate_model(
            model,
            setting_data['X_test'], setting_data['y_test'],
            class_names=setting_data['class_names'],
        )
        all_results[setting_name][arch] = metrics
        
        # Confusion matrix
        plot_confusion_matrix(
            setting_data['y_test'], metrics['y_pred'],
            class_names=setting_data['class_names'],
            title=f'Confusion Matrix — {arch} ({setting_name})',
            save_path=f'../outputs/{setting_name.replace(" ", "_")}_{arch}_cm.png'
        )

print('\n✓ All training complete!')

In [ ]:
# Build and display comparison table
comparison_df = build_comparison_table(all_results)
display_comparison(comparison_df, save_path='../outputs/comparison_table.csv')

### Analysis — Comparing Training Settings

*Write your 8-10 line analysis here comparing the three settings and explaining which performed best and why.*

> **Key points to cover:**
> - Baseline results: High accuracy but poor recall on minority classes
> - Under-sampling: Better balance but potential loss of information from majority classes
> - Under-sampling + Augmentation: Best overall performance
> - Effect on F1-score vs raw accuracy
> - Clinical implications: which metric matters most?

---
## Summary

This project demonstrates that **standard accuracy is misleading for imbalanced medical datasets**. By combining under-sampling + data augmentation + transfer learning with adaptive LR schedules, we can build models that perform meaningfully well across all classes, including rare but clinically critical conditions.